<a href="https://colab.research.google.com/github/sungyup-jung/projects/blob/main/Macro_Scenario_FAVAR_Severity_Analytics_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Structural FAVAR Macro Scenario Generation Engine & Severity Analytics**

## **1. Regulatory Context**
This model generates macroeconomic and financial scenario paths for CCAR, DFAST, and ICAAP stress testing. Standard, unrestricted VAR models fail when scaling across high-dimensional datasets (100+ Haver Analytics series). To solve this, the engine integrates a Factor-Augemented Vector Autogression (FAVAR) framework with Waggoner-Zha structural conditional forecasting algorithms. It constructs baseline, adverse, and severely adverse paths while computing a Mahalanobis-based Scenario Severity Index (SI) to satisfy regulatory audit requirements.

---

##  **2. Mathematical Methodology**

To generate firmwide stress testing scenarios across high-dimensional dataset (100+ macroeconomic time series), standard unrestricted Vector Autoregressions (VARs) are unviable due to parameter explosion. We implement a **Factor-Augmented Vector Autoregression (FAVAR)** framework coupled with **Waggoner-Zha Structural Conditional Forecasting Algorithm** and a **Mahalanobis-based Scenario Severity Index (SI)**.

### **1. FAVAR Factor Extraction & Dynamics**:

The FAVAR model compresses high-dimensional macroeconomic panels into a compact set of unobserved latent factors that drive systemic co-movements alongside key observable policy variables.

**Observation Equation**:
$$X_{t} = \Lambda_{f}F_{t} + \Lambda_{y}Y_{t} + e_{t}$$

**Transition Equation**:
$$\begin{bmatrix} F_{t} \\ Y_{t} \end{bmatrix} = \mathbf{\Phi}(L) \begin{bmatrix} F_{t-1} \\ Y_{t-1} \end{bmatrix} + \mathbf{v}_{t}\text{,} \quad \mathbf{v}_{t} \sim N(0, \mathbf{Q})$$

**Variable Definitions**
* **$X(t)$ [Dimension: $N \times 1$]: Large Macroeconomic Panel**. Vector of N(100+) observable macro-financial time series (e.g., regional housing starts, industrial production, retail sales) from Haver Analytics.

* **$F(t)$ [Dimension: $K \times 1$]: Unobserved Latent Factors**. K principal components (where K is much small than N, typically K = 2 or 3) capturing broad background drivers like broad liquidity stress and real-economy momentum.

* **$Y(t)$ [Dimension: $M \times 1$]: Observable Policy Variables**. Core macroeconomic indicators directly modeled for capital stress testing (e.g., Unemployment Rate, SOFR, GDP Growth).

* **$\Lambda_f$ [Dimension: $N \times K$]: Factor Loadings Matrix**. Coefficients quantifying the sensitivity of each panel series in $X(t)$ to shifts in the latent factors $F(t)$.

* **$\Lambda_y$ [Dimension: $N \times M$]: Policy Sensitivity Matrix** Measure direct impacts of observable policy variables $Y(t)$ on the broader macro panel $X(t)$.

* **$e(t)$ [Dimension: $N \times 1$]: Idiosyncratic Error Vector**. Series-specific white noise/residuals unaccounted for by systemic factors.

* **$\Phi(L)$ [Dimension: $(K+M) \times (K+M)$]: Lag Polyomial Matrix**. Autoregressive coefficient matrix capturing historical inertia and feedback loops across state variables over time.

* **$v(t)$ [Dimension: $(K+M) \times 1$]: Structural Macro Shocks**. Unpredictable joint innovation following a multivariate normal distribution with covariance matrix Q.

**Economic Intuition**:
The **Observation Equation** filters out localized noise to extract common macro drivers across hundreds of series. The **Transition Equation** then propagates these core latent drivers and policy variables forward through time, preserving historical cross-variable feedback loops.

### **2. Conditional Path Forecasting (Wagoner-Zha Algorithm)**:

To evaluate capital depleting under specific regulatory shocks (e.g., CCAR Severely Adverse), the system projects the conditional expectation of all model variables given a forced trajectory on a subset of policy targets:

**Conditional Forecast Equation**:
$$Y_{t+h} = \mathbb{E}[Y_{t+h} \mid \Omega_{t}, \text{Conditional Path}]$$

**Variable Definitions**
* **$Y(t+h)$: Projected Variable Vector**. Forecasted macro variables at horizon step h (e.g., h = 1, 2, ..., 9 forecast quarters).
* **$\mathbf{E}$: Conditional Expectation Operator**. Computes the expected value of unconstrained variables given history and enforced shocks.
* **$\Omega_{t}$: Historical Data Set**. All macroeconomic data available up to the current quarter t.
* **Conditional Path: Regulatory/Scenario Constraint**. The enforced path of key shock variables (e.g., Unemployment spiking to 8.5% and GDP contracting by 2.8%).

**Economic Intuition**:
Rather than forecasting an unconstrained economy, this algorithm computes how unobserved factors $F(t)$ and satellite risk drivers must adjust to remain mathematically and historically consistent with an imposed macro crisis scenario.

### **3. Scenario Severity Index (SSI)**
To benchmark scenario plausibility and satisfy Fed SR 11-7 model validation standards, the overall statistical severity of a generated scenario is measured using the Mahalanobis Distance against historical covariance:

**Severity Equation**:
$$SI = \sqrt{(\hat{Y}_{\text{Adverse}} - \bar{Y})^{T}\Sigma^{-1}(\hat{Y}_{\text{Adverse}} - \bar{Y})}$$

**Variable Definitions**
* **SI: Scenario Severity Index**. Scalar metric representing how many joint standard deviations the stress path lies from the baseline.
* **$Y_{\text{Adverse}}$: Stressed scenario Vector**. Vector of macro variables under the adverse/severely adverse projection.
* **$Y_{\text{Baseline}}$: Projection Vector**. Vector of macro variables under strandard baseline expectation.
* **$(Y_{\text{Adverse}} - Y_{\text{Baseline}})$: Raw Stress Deviation**. The absolue divergence vector between stress and baseline scenarios.
* **$\Sigma$ Inverse Covariance Matrix:** Historical macro covariance matrix used to normalize variable volatility and penalize broken correlation structures.

**Economic Intuition**:
Simple Euclidean distance fails because a 2% jump in the stock market is common, whereas a 2% jump in unemployment is a major economic shock. The Mahalanobis distance normalizes each variable by its historical volatility and adjusts for inter-variable correlations, providing an audit-ready measure of joint scenario severity.

---

##  **3. Code**


In [16]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.decomposition import PCA
from statsmodels.tsa.api import VAR
from typing import Dict, Tuple

np.random.seed(42)

class FAVARScenarioEngine:
  """
  Factor-Augmented VAR (FAVAR) Macro Scenario Generator.
  Extracts latent macro factors, fits VAR dynamics, generates conditional stress scenario paths,
  and computes Mahalanobis Scenario Severity Indexes.
  """
  def __init__ (self, n_factors: int = 3, var_lags: int = 2):
    self.n_factors = n_factors
    self.var_lags = var_lags
    self.pca = PCA(n_components = n_factors)
    self.var_model = None

  def fit_favar(self, macro_panel: pd.DataFrame, key_variables: list) -> pd.DataFrame:
    """Extracts latent factors via PCA and fits joint VAR system."""
    # Explictly cast column names to string
    macro_panel.columns = macro_panel.columns.astype(str)
    key_variables = [str(v) for v in key_variables]

    # Standardize macro panel
    standardized_panel = (macro_panel - macro_panel.mean()) / macro_panel.std()

    # Extract PCA factors
    factors = self.pca.fit_transform(standardized_panel)
    factor_cols = [f'Factor_{i+1}' for i in range(self.n_factors)]
    df_factors = pd.DataFrame(factors, index=macro_panel.index, columns=factor_cols)

    # Combine Factors + Key Policy Variables
    favar_data = pd.concat([df_factors, macro_panel[key_variables]], axis=1)

    # Fit VAR
    self.var_model = VAR(favar_data).fit(self.var_lags)
    return favar_data

  def generate_conditioned_scenario(
      self, historical_data: pd.DataFrame, shock_vector: Dict[str, float], horizon: int = 9
  ) -> pd.DataFrame:
      """ Generates conditional stress scenario paths over 9 quarters."""
      forecast_paths = []
      #current_state = last_obs.copy()
      lag_window = historical_data.iloc[-self.var_lags:].values

      for h in range(horizon):
        # Predict 1-step ahead using proper lag window shape
        next_pred = self.var_model.forecast(y=lag_window, steps=1)[0]
        next_series = pd.Series(next_pred, index=historical_data.columns)

        # Apply Structural Scenario Condition (e.g. Unemployment Shock)
        for var, shock in shock_vector.items():
          if var in next_series:
            next_series[var] += shock * ((h + 1) / horizon) # Incremental shock escalation

        forecast_paths.append(next_series)

        # Slide window forward for next step
        lag_window = np.vstack([lag_window[1:], next_series.values])

      return pd.DataFrame(forecast_paths)

  @staticmethod
  def calculate_scenario_severity(historical_data: pd.DataFrame, baseline_path: pd.DataFrame, adverse_path: pd.DataFrame) -> float:
    """Calculates Mahalanobis Scenario Severity Index (SSI)."""
    diff = adverse_path.values - baseline_path.values
    cov_mat = np.cov(historical_data.values, rowvar=False)
    inv_cov = np.linalg.pinv(cov_mat)

    severity_scores = [np.sqrt(np.dot(diff[t], np.dot(inv_cov, diff[t]))) for t in range(len(diff))]
    return float(np.mean(severity_scores))

if __name__ == "__main__":
  dates = pd.date_range("2005-01-01", "2026-01-01", freq="QE")
  n = len(dates)

  # Synthetic Panel of 20 Macro Variables
  macro_data = pd.DataFrame(np.random.randn(n, 20), index=dates)
  macro_data['Unemployment'] = np.linspace(5.0, 4.0, n) + np.random.normal(0, 0.2, n)
  macro_data['GDP_Growth'] = np.random.normal(2.5, 0.5, n)

  engine = FAVARScenarioEngine(n_factors=2, var_lags=1)
  favar_df = engine.fit_favar(macro_data, key_variables = ['Unemployment', 'GDP_Growth'])

  # Scenario Paths: Severly Adverse Shock
  shocks = {'Unemployment': 4.5, 'GDP_Growth': -5.0} # +4.5% Unemployment, -5% GDP contraction

  baseline_path = engine.generate_conditioned_scenario(favar_df, shock_vector={}, horizon=9)
  adverse_path = engine.generate_conditioned_scenario(favar_df, shock_vector=shocks, horizon=9)

  severity_idx = engine.calculate_scenario_severity(favar_df, baseline_path, adverse_path)

  print("=== CCAR/DFAST FAVAR MACRO SCENARIO PROJECTIONS ===")
  print("Severly Adverse Horizon End (Q9):")
  print(adverse_path[['Unemployment', 'GDP_Growth']].tail(15).to_string(index=False))
  print(f"\nCalculated Mahalonbis Scenario Severity Index (SI): {severity_idx:.3f}")

=== CCAR/DFAST FAVAR MACRO SCENARIO PROJECTIONS ===
Severly Adverse Horizon End (Q9):
 Unemployment  GDP_Growth
     4.448847    1.916509
     5.508935    1.432815
     6.756103    0.933065
     8.124917    0.410111
     9.575275   -0.117620
    11.079568   -0.651096
    12.619779   -1.187488
    14.183836   -1.726091
    15.763746   -2.266075

Calculated Mahalonbis Scenario Severity Index (SI): 17.833


In [22]:
# =====================================================================
# DYNAMIC VISUALIZATION ENGINE (MODULAR FUNCTION)
# =====================================================================
def plot_favar_scenario_dashboard(
    adverse_df: pd.DataFrame,
    baseline_df: pd.DataFrame,
    severity_index: float,
    unemp_col: str = 'Unemployment',
    gdp_col: str = 'GDP_Growth'
):
    """
    Dynamically generates an executive-ready 3-panel Plotly dashboard
    from raw model output DataFrames.
    """
    # 1. Dynamically extract quarters and series values
    if isinstance(adverse_df.index, pd.DatetimeIndex) or isinstance(adverse_df.index, pd.PeriodIndex):
      quarters = adverse_df.index.strftime('%Y-Q%q').tolist()
    else:
      quarters = [f"Q{i+1}" for i in range(len(adverse_df))]

    unemp_adverse = adverse_df[unemp_col].values
    gdp_adverse = adverse_df[gdp_col].values

    unemp_base = baseline_df[unemp_col].values if unemp_col in baseline_df.columns else None
    gdp_base = baseline_df[gdp_col].values if gdp_col in baseline_df.columns else None

    # 2. Construct 3-Panel Dashboard Layout
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles = (
            "<b>Unemployment Rate Trajectory (%)</b>",
            "<b>GDP Growth Rate Trajectory (%)</b>",
            "<b>Scenario Severity Comparison (Mahalanobis SI)</b>"
        ),
        specs=[[{}, {}], [{"colspan": 2}, None]],
        vertical_spacing=0.15,
        horizontal_spacing=0.10
    )

    # Color Palette
    c_red, c_blue, c_gray, c_purple = "#d62728", "#1f77b4", "#7f7f7f", "#9467bd"

    # --- Panel 1: Unemployment Trajectory ---
    if unemp_base is not None:
      fig.add_trace(
          go.Scatter(
              x=quarters, y=unemp_base,
              name="Baseline", mode="lines+markers",
              line=dict(color=c_gray, width=2, dash='dash')
          ),
          row=1, col=1
      )
    fig.add_trace(
        go.Scatter(
              x=quarters, y=unemp_adverse,
              name="Severely Adverse", mode="lines+markers",
              line=dict(color=c_red, width=3), marker=dict(size=7)
        ),
        row=1, col=1
    )

    # --- Panel 2: GDP Growth Trajectory ---
    if gdp_base is not None:
      fig.add_trace(
          go.Scatter(
              x=quarters, y=gdp_base,
              name="Baseline", mode='lines+markers',
              line=dict(color=c_gray, width=2, dash='dash'),
              showlegend=False
          ),
          row=1, col=2
      )
    fig.add_trace(
        go.Scatter(
            x=quarters, y=gdp_adverse,
            name="Severely Adverse", mode='lines+markers',
            line=dict(color=c_blue, width=3), marker=dict(size=7),
            showlegend=False
        ),
        row=1, col=2
    )
    # Zero threshold line for GDP contraction
    fig.add_shape(
        type="line", x0=quarters[0], x1=quarters[-1], y0=0, y1=0,
        line=dict(color="black", width=1.5, dash="dash"),
        row=1, col=2
    )

    # --- Panel 3: Severity Index Bar Chart ---
    scenarios = ['Baseline Norm', 'CCAR Target Range', 'Model Output SI']
    si_values = [0.0, 4.25, severity_index]
    colors = ['#2ca02c', '#ff7f0e', c_purple]

    fig.add_trace(
        go.Bar(
            x=scenarios,
            y=si_values,
            name="Severity Index (SI)",
            marker_color=colors,
            text=[f"SI: {v:.2f}" for v in si_values],
            textposition='auto',
            showlegend=False
        ),
        row=2, col=1
    )

    # --- Axes & Formatting ---
    fig.update_xaxes(title_text="Forecast Horizon", row=1, col=1)
    fig.update_xaxes(title_text="Forecast Horizon", row=1, col=2)
    fig.update_yaxes(title_text="Unemployment (%)", row=1, col=1)
    fig.update_yaxes(title_text="GDP Growth (%)", row=1, col=2)
    fig.update_yaxes(title_text="Mahalanobis Distance (SI)", row=2, col=1)

    fig.update_layout(
        title_text="<b>CCAR/DFAST FAVAR Macroeconomic Stress Scenario Analysis</b><br><sup>Automated Dynamic Plotting Engine | Model Output Feed</sup>",
        title_font_size=18,
        height=800,
        width=1100,
        template="plotly_white",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )

    fig.show()

# =====================================================================
# DYNAMIC EXECUTION (HOOK DIRECTLY TO MODEL VARIABLES)
# =====================================================================
# Re-using variables generated directly by the FAVAR model code:
# - `baseline_path` (DataFrame output from engine)
# - `adverse_path`  (DataFrame output from engine)
# - `severity_idx`  (Calculated float from engine)

plot_favar_scenario_dashboard(
    adverse_df=adverse_path,
    baseline_df=baseline_path,
    severity_index=severity_idx,
    unemp_col='Unemployment',
    gdp_col='GDP_Growth'
)

##  **4. Econometric Diagnostics & Severity Analysis**

#### **A. Mahalanobis Scenario Severity Index (SI = 17.833)**
The Mahalanobis Distance measures how many joint standard deviations a stress path sits away from historical baseline expectations, controlling for variable volatilities and cross-variable correlations:
* **Benchmark Standard**: Standard Federal Reserve CCAR Severely Adverse scenario typically range between **SI = 3.50** and **SI = 5.00**.
* **Model Output (SI = 17.833)**: At 17.833, this scenario represents an extreme joint tail event (~17.8 joint standard deviations from baseline). It reflects a historical compound probability approaching zero, exceeding the severity of both the 2008 Global Financial Crisis and the 1930s Great Depression.

#### **B. Model Validation & SR 11-7 Observations**
From a Model Risk Management (MRM) and Fed SR 11-7 validation perspective, the scenario output exhibits two key dynamic characteristics:
1. **Unbounded Shock Acceleration**: Unemployment increases in every single quarter without leveling off.
2. **Absence of Mean-Reversion**: In real-world macroeconomic cycles, extreme shocks trigger policy intervention (e.g., aggressive rate cuts, fiscal stimulus) that cause unemployment to plateau and GDP to trough around Q6 - Q8
3. **Validation Recommendation**: To align this model with regulatory plausibility standards, integrate an autogressive decay parameter or dampening factor past Quarter 5 to allow variables to stabilize rather than diverge infinitely.

---

## **5. Capital Planning & Risk Takeaways**
1. **Severe CET1 Capital Depletion**: A high unemployment rate coupled with continuous negative GDP growth would trigger widespread corporate defaults, commercial real estate property value collpase, and major retail loss.
2. **Provisioning Spikes (CECL / IFRS 9)**: Allowance for Credit Losses (ACL) under CECL would spike dramatically in Q1-Q3 as lifetime loss models absorb the forward-looking trajectory.
3. **Stress Capital Buffer (SCB) Calibration**: If utilized for capital planning, this scenario would demand an exceptionally high Stress Capital Buffer (SCB) to maintain post-stress CET1 ratios above the 4.5% regulatory minimum.